# cfd10 — Colab teacher training

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** (or L4).

Then **Runtime -> Run all**. Cell 3 auto-locates your `raw_v16` data in Drive (prefers a folder literally named `raw_v16`). Cell 5 prints the scorecard to paste back into the chat.

In [ ]:
# Cell 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Clone the (public) repo and install deps
from pathlib import Path
REPO_DIR = Path('/content/cfd10')
REPO_URL = 'https://github.com/Sovenski/cfd10.git'
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only -q
# Core deps from pyproject; does NOT touch Colab's GPU torch (torch is a dev-only extra).
!pip install -q -e .
!git rev-parse --short HEAD

In [ ]:
# Cell 3 - Auto-locate the raw_v16 data in Drive, link it, confirm GPU
import os, glob, torch
found = !find /content/drive/MyDrive -maxdepth 6 -iname 'SP_SPX, 1D*.csv' 2>/dev/null
cands = [os.path.dirname(p) for p in found]
pref = [d for d in cands if d.rstrip('/').endswith('raw_v16')]
DATA_DIR = (pref or cands or [''])[0]   # prefer a folder literally named raw_v16
# If wrong/empty, set it manually (browse the left Files panel -> drive/MyDrive):
# DATA_DIR = '/content/drive/MyDrive/cfd9/data/raw_v16'
print('candidates:', list(cands)[:5])
assert DATA_DIR, ('Auto-find failed. Set DATA_DIR manually above. If the folder is under '
                  '"Shared with me", right-click it in Drive -> Add shortcut to Drive -> My Drive, then re-run.')
link = '/content/cfd10/data/raw_v16'
os.makedirs('/content/cfd10/data', exist_ok=True)
if os.path.islink(link) or os.path.exists(link):
    os.remove(link)   # clear any stale/broken link from a previous attempt
os.symlink(DATA_DIR, link)
n_csv = len(glob.glob(link + '/*.csv'))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-ONLY (switch runtime to GPU!)'
print(f'data: {DATA_DIR}\n{n_csv} CSVs | device: {gpu}')
assert n_csv > 50, f'Only {n_csv} CSVs — expected ~80. Check DATA_DIR points at the clean raw_v16 folder.'

In [ ]:
# Cell 4 - Train the TCN teacher (beat-the-GBDT-baseline gate). ~minutes on a T4/L4.
!python pipeline/fit_teacher.py --epochs 30

In [ ]:
# Cell 5 - Print the teacher scorecard (copy this back into the chat) + back it up to Drive
import os, shutil
OUT, DRIVE_OUT = '/content/cfd10/outputs', '/content/drive/MyDrive/cfd10_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in ('teacher_pooled_scorecard.md', 'baseline_pooled_scorecard.md'):
    if os.path.exists(f'{OUT}/{f}'):
        shutil.copy(f'{OUT}/{f}', f'{DRIVE_OUT}/{f}')
print(open(f'{OUT}/teacher_pooled_scorecard.md', encoding='utf-8').read())

In [ ]:
# Cell 6 (optional) - GBDT pooled baseline + label QA for reference
!python pipeline/fit_pooled.py
print(open('/content/cfd10/outputs/baseline_pooled_scorecard.md', encoding='utf-8').read())